# Task 2 - End-to-End ML Pipeline

## Install

In [ ]:
!pip -q install scikit-learn pandas joblib

## Load Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report
import joblib

# Upload WA_Fn-UseC_-Telco-Customer-Churn.csv
from google.colab import files
uploaded=files.upload()
df=pd.read_csv(next(iter(uploaded)))
df.head()

## Preprocess

In [ ]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
X=df.drop('Churn',axis=1)
y=df['Churn'].map({'No':0,'Yes':1})
cat=X.select_dtypes(include='object').columns
num=X.select_dtypes(exclude='object').columns
pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer()),('sc',StandardScaler())]),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## Logistic Regression + GridSearch

In [ ]:
pipe=Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1000))])
param={'clf__C':[0.1,1,10]}
grid=GridSearchCV(pipe,param_grid=param,cv=3,scoring='accuracy')
grid.fit(X_train,y_train)
p=grid.predict(X_test)
print(accuracy_score(y_test,p))
print(classification_report(y_test,p))

## Random Forest

In [ ]:
rf=Pipeline([('pre',pre),('clf',RandomForestClassifier(random_state=42))])
rf.fit(X_train,y_train)
pr=rf.predict(X_test)
print(accuracy_score(y_test,pr))

## Save Pipeline

In [ ]:
joblib.dump(grid.best_estimator_,'churn_pipeline.joblib')